# ANCOVA

# Baseline

In [7]:
# Comment out the following line to run the notebook in CPU mode
%load_ext cudf.pandas
import plotly.express as px
import pandas as pd
# Suppress warnings
import warnings

warnings.filterwarnings("ignore")

The cudf.pandas extension is already loaded. To reload it, use:
  %reload_ext cudf.pandas


### Significance

Significance is the fraction of statistical test that are significant (i.e. p-value < 0.05) among the 26 MCA repetitions.

#### Uncorrected

## Subcortical Volume

### Uncorrected

In [46]:
import plotly.graph_objects as go
# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

regions = ["Left-Thalamus", "Left-Caudate", "Left-Putamen", "Left-Pallidum", "Left-Hippocampus", "Left-Amygdala", "Left-Accumbens-area",  
           "Right-Thalamus", "Right-Caudate", "Right-Putamen", "Right-Pallidum", "Right-Hippocampus", "Right-Amygdala", "Right-Accumbens-area"] 

baseline_subcortical_volume = pd.read_parquet("ancova/ancova_baseline_subcortical_volume.parquet")
baseline_subcortical_volume['significant'] = baseline_subcortical_volume['p-unc'] < 0.05
baseline_subcortical_volume_avg = baseline_subcortical_volume.groupby("region").agg({"F": "mean", "significant": "mean"}).reset_index()

ieee_data = pd.read_csv("ancova_ieee/ancova_baseline_subcortical_volume.csv")

ieee_data.sort_values(by=["region"], inplace=True)
ieee_data['fs_version'] = "IEEE"

# Filter out the regions that are not in the data
baseline_subcortical_volume = baseline_subcortical_volume[baseline_subcortical_volume["region"].isin(regions)]
baseline_subcortical_volume_avg = baseline_subcortical_volume_avg[baseline_subcortical_volume_avg["region"].isin(regions)]
ieee_data = ieee_data[ieee_data["region"].isin(regions)]

# Create a position mapping dictionary
region_to_position = {region: i for i, region in enumerate(regions)}

# Add position columns to each dataframe
baseline_subcortical_volume['position'] = baseline_subcortical_volume['region'].map(region_to_position)
baseline_subcortical_volume_avg['position'] = baseline_subcortical_volume_avg['region'].map(region_to_position)
ieee_data['position'] = ieee_data['region'].map(region_to_position)

# Sort by position to ensure correct order
baseline_subcortical_volume.sort_values(by="position", inplace=True)
baseline_subcortical_volume_avg.sort_values(by="position", inplace=True)
ieee_data.sort_values(by="position", inplace=True)

# Add small value to significant to make it visible on the plot
# baseline_subcortical_volume_avg["significant"] = baseline_subcortical_volume_avg["significant"] + 0.02

fig = go.Figure()

fig.update_layout(title_text="Group differences (PD vs HC) in subcortical volumes at baseline")

# Use the position columns for x-axis placement
bar_offset = -0.1  # Shift bars left
violin_offset = 0.15  # Shift violins right

blue = px.colors.qualitative.Plotly[4]

# Create a bar chart with offset x positions
bar = go.Bar(
    x=baseline_subcortical_volume_avg["position"] + bar_offset,
    y=baseline_subcortical_volume_avg["significant"],
    name="Significant Ratio",
    marker=dict(color=blue),
    showlegend=True,
    width=0.2,
    yaxis="y",
    customdata=baseline_subcortical_volume_avg["region"],
    hovertemplate="Region: %{customdata}<br>Significant Ratio: %{y:.2f}<extra></extra>"
)

violin = go.Violin(
    x=baseline_subcortical_volume["position"] + violin_offset,
    y=baseline_subcortical_volume["F"],
    yaxis="y2",
    box_visible=False,
    line_color="black",
    meanline_visible=True,
    fillcolor='black',    
    opacity=0.4,
    showlegend=True,
    name="F-value",
    spanmode="hard",
    points=False,
    customdata=baseline_subcortical_volume["region"],
    hovertemplate="Region: %{customdata}<br>F: %{y:.3f}<extra></extra>"
)

ieee = go.Scatter(
    x=ieee_data["position"] + violin_offset,
    y=ieee_data["F"],
    mode='markers',
    yaxis="y2",
    marker=dict(color='black'),
    showlegend=True,
    name="IEEE",
    customdata=ieee_data["region"],
    hovertemplate="Region: %{customdata}<br>F: %{y:.3f}<extra></extra>"
)

fig.add_trace(bar)
fig.add_trace(ieee)
fig.add_trace(violin)

# Set x-axis ticks and labels
fig.update_xaxes(
    title_text="Region",
    tickvals=list(range(len(regions))),
    ticktext=regions,
    tickangle=45
)

fig.update_yaxes(title_text="Group Differences Average")

fig.update_layout(
    yaxis=dict(
        title="Significant Group Differences Ratio", 
        range=[-0.05, 1.05],
        showgrid=True, 
        dtick=0.1  # Add grid lines each 0.1
    ),
    yaxis2=dict(
        title="F-value Distribution", 
        overlaying="y", 
        side="right", 
        range=[-1.25, 21.25],
        showgrid=False,
        zeroline=False
    ),
    legend=dict(
        x=0.5,
        y=1.15,
        orientation="h",
        xanchor="center"
    ),
    margin=dict(b=150),  # Add bottom margin for tilted labels
)

xrange = np.arange(0, 1.1, 0.1)
xrange_text = [str(round(x, 1)) for x in xrange]

fig.update_layout(yaxis=dict(tickvals=xrange, ticktext=xrange_text))
fig.update_layout(yaxis2=dict(tickvals=[0, 5, 10, 15, 20], ticktext=[0, 5, 10, 15, 20]))

fig.show()

In [49]:
import plotly.graph_objects as go
import numpy as np

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

regions = ["Left-Thalamus", "Left-Caudate", "Left-Putamen", "Left-Pallidum", "Left-Hippocampus", "Left-Amygdala", "Left-Accumbens-area",  
           "Right-Thalamus", "Right-Caudate", "Right-Putamen", "Right-Pallidum", "Right-Hippocampus", "Right-Amygdala", "Right-Accumbens-area"] 

longitudinal_subcortical_volume = pd.read_parquet("ancova/ancova_long_subcortical_volume.parquet")
longitudinal_subcortical_volume['significant'] = longitudinal_subcortical_volume['p-unc'] < 0.05
longitudinal_subcortical_volume_avg = longitudinal_subcortical_volume.groupby("region").agg({"F": "mean", "significant": "mean"}).reset_index()

ieee_data = pd.read_csv("ancova_ieee/ancova_longitudinal_subcortical_volume.csv")

ieee_data.sort_values(by=["region"], inplace=True)
ieee_data['fs_version'] = "IEEE"

# Filter out the regions that are not in the data
longitudinal_subcortical_volume = longitudinal_subcortical_volume[longitudinal_subcortical_volume["region"].isin(regions)]
longitudinal_subcortical_volume_avg = longitudinal_subcortical_volume_avg[longitudinal_subcortical_volume_avg["region"].isin(regions)]
ieee_data = ieee_data[ieee_data["region"].isin(regions)]

# Create a position mapping dictionary
region_to_position = {region: i for i, region in enumerate(regions)}

# Add position columns to each dataframe
longitudinal_subcortical_volume['position'] = longitudinal_subcortical_volume['region'].map(region_to_position)
longitudinal_subcortical_volume_avg['position'] = longitudinal_subcortical_volume_avg['region'].map(region_to_position)
ieee_data['position'] = ieee_data['region'].map(region_to_position)

# Sort by position to ensure correct order
longitudinal_subcortical_volume.sort_values(by="position", inplace=True)
longitudinal_subcortical_volume_avg.sort_values(by="position", inplace=True)
ieee_data.sort_values(by="position", inplace=True)

fig = go.Figure()

fig.update_layout(title_text="Longitudinal group differences (PD vs HC) in the rate of change in subcortical volumes")

# Use the position columns for x-axis placement
bar_offset = -0.2  # Shift bars left
violin_offset = 0.2  # Shift violins right

blue = px.colors.qualitative.Plotly[4]

# Create a bar chart with offset x positions
bar = go.Bar(
    x=longitudinal_subcortical_volume_avg["position"] + bar_offset,
    y=longitudinal_subcortical_volume_avg["significant"],
    name="Significant Ratio",
    marker=dict(color=blue),
    showlegend=True,
    width=0.25,
    yaxis="y",
    customdata=longitudinal_subcortical_volume_avg["region"],
    hovertemplate="Region: %{customdata}<br>Significant Ratio: %{y:.2f}<extra></extra>"
)

violin = go.Violin(
    x=longitudinal_subcortical_volume["position"] + violin_offset,
    y=longitudinal_subcortical_volume["F"],
    yaxis="y2",
    box_visible=False,
    line_color="black",
    meanline_visible=True,
    fillcolor='black',    
    opacity=0.4,
    showlegend=True,
    name="F-value",
    points=False,
    spanmode='hard',
    customdata=longitudinal_subcortical_volume["region"],
    hovertemplate="Region: %{customdata}<br>F: %{y:.3f}<extra></extra>"
)

ieee = go.Scatter(
    x=ieee_data["position"] + violin_offset,
    y=ieee_data["F"],
    mode='markers',
    yaxis="y2",
    marker=dict(color='black'),
    showlegend=True,
    name="IEEE",
    customdata=ieee_data["region"],
    hovertemplate="Region: %{customdata}<br>F: %{y:.3f}<extra></extra>"
)

fig.add_trace(bar)
fig.add_trace(ieee)
fig.add_trace(violin)

# Set x-axis ticks and labels
fig.update_xaxes(
    title_text="Region",
    tickvals=list(range(len(regions))),
    ticktext=regions,
    tickangle=45
)

fig.update_yaxes(title_text="Group Differences Average")

fig.update_layout(
    yaxis=dict(
        title="Significant Partial Correlation Ratio", 
        range=[-0.05, 1.05],
        showgrid=True,
        dtick=0.1  # Add grid lines each 0.1
    ),
    yaxis2=dict(
        title="F-value Distribution", 
        overlaying="y", 
        side="right", 
        range=[-4, 71], 
        showgrid=False,
        zeroline=False,
    ),
    legend=dict(
        x=0.5,
        y=1.15,
        orientation="h",
        xanchor="center"
    ),
    margin=dict(b=150),  # Add bottom margin for tilted labels
)

xrange = np.arange(0, 1.1, 0.1)
xrange_text = [str(round(x, 1)) for x in xrange]

fig.update_layout(yaxis=dict(tickvals=xrange, ticktext=xrange_text))
fig.update_layout(yaxis2=dict(tickvals=[0, 20, 40, 60], ticktext=[0, 20, 40, 60]))

fig.show()